# Column Generation (Basics, Dual Stabilization, price-and-branch) — Before, Principle, How-to, Effects

For models with an exponentially large number of "possible patterns" (cutting patterns, shift patterns, routes, etc.), writing down and solving a **compact formulation** that enumerates all patterns is not always feasible.
**Column generation** avoids this by alternately solving a Restricted Master Problem (RMP: an LP with only the columns currently available) and a **pricing problem** (a small optimization problem to find one improving column), generating only the necessary columns on the fly.

This notebook traces column generation in the following pattern:

1. **Before (Issue)** — The weakness of the compact formulation (Kantorovich) that enumerates all patterns.
2. **Principle** — How the LP bound monotonically tightens through RMP↔pricing iterations (empirical).
3. **How-to (Application)** — Verifying `mk.column_generation` / `mk.price_and_branch` with a minimal setup.
4. **Effects (Before/After)** — Comparing the compact formulation vs. the restricted master problem (integer) after column generation.

The subject is **Cutting Stock** (`samples/packing_and_cutting/cutting_stock.py`).
A classic column generation problem where we cut items (width, demand) from rolls of width 100, minimizing the number of rolls used.

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/packing_and_cutting"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import cutting_stock as cs

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. Before (Issue) — Naively Solving the Compact Formulation (Kantorovich)

`cs.build_compact()` is a compact formulation that has `y_r` (whether to use roll `r` or not) and pattern `a_{i,r}` for each roll `r`. Because it does not have the concept of a pattern and directly formulates "how many of which item to pack into which roll," it becomes **symmetric with respect to roll swapping** (equivalent no matter which roll index is used).
Diagnosing with `mk.analyze` reveals that while symmetry (reference information, handled automatically by SCIP) is detected, it still takes **over 5000 nodes** to solve even after processing that symmetry.

In [2]:
report = mk.analyze(cs.build_compact, name="cutting-stock-compact", time_limit=20)
print(report.summary())
print("探索ノード数:", report.metrics.get("nodes"), " 最終gap:", report.metrics.get("gap"),
      " 対称群の数:", report.metrics.get("n_sym_groups"),
      " 最大対称群サイズ:", report.metrics.get("largest_sym_group"))

[cutting-stock-compact] 検出症状 1件:
  [good] 入替可能な変数群(対称性)を検出(参考情報) -> 通常は対応不要(SCIPが自動処理)。usesymmetryを無効化した運用でのみ辞書式除去が有効
探索ノード数: 5231  最終gap: 0.0  対称群の数: 7  最大対称群サイズ: 60


Up to 60 rolls enter the model as a symmetry group where they can be swapped with each other. SCIP automatically detects symmetry to reduce branching (a `good` finding), but it still requires thousands of nodes to solve.
This is why column generation, which **treats patterns as columns and generates only the necessary columns**, is used in practice for this type of model.

## 2. Principle — The Process of Tightening the LP Bound via RMP↔pricing Iterations (Empirical)

Solve the Restricted Master Problem (RMP) $\min \sum_p \lambda_p \ \text{s.t.}\ \sum_p \mathrm{col}_p \lambda_p \ge \mathrm{demand},\ \lambda\ge0$ to obtain duals $\pi$.
Under $\pi$, find one cutting pattern with the highest value using a **pricing knapsack** problem
$\max \sum_i \pi_i a_i \ \text{s.t.}\ \sum_i w_i a_i \le W$. If the reduced cost (`pricing value > 1`) is negative,
add it to the RMP. Repeat this process. When the pricing value becomes 1 or less, there are no improving patterns = LP optimal.

```
  RMP min Σλ s.t. Σcol·λ>=demand ──Dual π──▶ pricing max Σπ_i·a_i s.t. Σw_i·a_i<=W
       ▲                                                    │
       └──────── Add new column with reduced cost < 0 (pricing value > 1) ──┘
  Terminate when pricing value <= 1 (no improving columns) = LP optimal
```

Initial columns start with "trivial patterns packed with only each individual item."

In [3]:
def pricing(duals):
    kp = Model(); kp.hideOutput()
    a = [kp.addVar(vtype="I", lb=0, name=f"a{i}") for i in range(cs.N_ITEMS)]
    kp.addCons(quicksum(cs.WIDTHS[i] * a[i] for i in range(cs.N_ITEMS)) <= cs.W)
    kp.setObjective(quicksum(duals[i] * a[i] for i in range(cs.N_ITEMS)), "maximize")
    kp.optimize()
    return [round(kp.getVal(v)) for v in a], kp.getObjVal()


init_cols = [[cs.W // cs.WIDTHS[i] if j == i else 0 for j in range(cs.N_ITEMS)]
             for i in range(cs.N_ITEMS)]
rhs = [float(d) for d in cs.DEMANDS]

res = mk.column_generation(rhs, init_cols, pricing, alpha=0.0)
print(f"GG LP境界={res['lp_bound']:.3f}  反復={len(res['history'])}  生成列数={res['n_cols']}")

GG LP境界=23.545  反復=8  生成列数=13


In [4]:
def count_patterns(widths, W):
    ws = sorted(set(widths))
    def rec(idx, rem):
        if idx == len(ws):
            return 1
        return sum(rec(idx + 1, rem - k * ws[idx]) for k in range(rem // ws[idx] + 1))
    return rec(0, W) - 1

total_patterns = count_patterns(cs.WIDTHS, cs.W)
material_lb = sum(cs.WIDTHS[i] * cs.DEMANDS[i] for i in range(cs.N_ITEMS)) / cs.W

its = [h["iter"] for h in res["history"]]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("RMPのLP境界の収束(列を追加するたび締まる)",
                    "pricing値(1.0以下で改善列なし=収束)"))
fig.add_trace(go.Scatter(x=its, y=[h["lp_bound"] for h in res["history"]], mode="lines+markers",
    line=dict(color=C["s1"], width=2), marker=dict(size=7, color=C["s1"]),
    name="LP境界", hovertemplate="反復%{x}<br>LP境界 %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_hline(y=material_lb, line=dict(color=C["muted"], width=2, dash="dash"),
              annotation_text=f"材料下界 {material_lb:.2f}", row=1, col=1,
              annotation_font=dict(color=C["ink2"], size=11))
fig.add_trace(go.Scatter(x=its, y=[h["pricing_val"] for h in res["history"]], mode="lines+markers",
    line=dict(color=C["s2"], width=2), marker=dict(size=7, color=C["s2"]),
    name="pricing値", hovertemplate="反復%{x}<br>pricing値 %{y:.3f}<extra></extra>"), row=1, col=2)
fig.add_hline(y=1.0, line=dict(color=C["warn"], width=2, dash="dash"),
              annotation_text="収束閾値1.0", row=1, col=2,
              annotation_font=dict(color=C["ink2"], size=11))
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380, showlegend=False,
    margin=dict(l=50, r=20, t=56, b=44))
fig.update_xaxes(title_text="列生成の反復", gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

The LP bound monotonically tightens until it reaches the material lower bound (the theoretically best LP lower bound), and when the pricing value drops to 1.0, it stops, judging that there are "no improving patterns." Looking at the ratio of generated columns to the **total feasible patterns**, the essence of column generation ("generating only the necessary columns on the fly") becomes visible in the numbers.

In [5]:
gen = res["n_cols"]
print(f"総実行可能パターン数: {total_patterns}")
print(f"列生成が生成した列数: {gen} ({gen/total_patterns*100:.1f}%)")
print(f"GG LP境界={res['lp_bound']:.3f}  材料下界={material_lb:.3f}  "
      f"(ほぼ一致 = 全列挙しなくても最良のLP下界に到達)")

総実行可能パターン数: 131
列生成が生成した列数: 13 (9.9%)
GG LP境界=23.545  材料下界=23.520  (ほぼ一致 = 全列挙しなくても最良のLP下界に到達)


## 3. How-to (Application) — `mk.column_generation` / `mk.price_and_branch`

The only problem-specific part is `pricing_fn(duals) -> (column, value)`. Here, we verify the API contract with a minimal setup (the same pricing as in Section 2, but we only observe that the LP bound after convergence is valid as a lower bound for the true integer optimal).

In [6]:
res_min = mk.column_generation(rhs, init_cols, pricing, alpha=0.0)
print(f"lp_bound={res_min['lp_bound']:.3f} は整数最適の下界(<= 真の整数最適)であるべき: "
      f"{res_min['lp_bound'] <= 24.0 + 1e-6}")

bnp = mk.price_and_branch(rhs, init_cols, pricing, alpha=0.0)
optimal_proved = abs(bnp["lp_lb"] - bnp["int_obj"]) < 1e-6
print(f"price_and_branch: lp_lb={bnp['lp_lb']}  int_obj={bnp['int_obj']:.3f}  "
      f"lp_lb==int_obj(最適性証明)={optimal_proved}")

lp_bound=23.545 は整数最適の下界(<= 真の整数最適)であるべき: True
price_and_branch: lp_lb=24  int_obj=24.000  lp_lb==int_obj(最適性証明)=True


`price_and_branch` solves the integer restricted master problem using **only** the columns already generated. This guarantees **only an upper bound** (the true integer optimal $\le$ this solution is not necessarily true). Optimality is proven only when `lp_lb == int_obj` holds — in this subject, it holds at 24 rolls.

## 4. Effects (Before/After) — Compact Formulation vs. Restricted Master Problem After Column Generation

We pass (a) the compact formulation from Section 1 (which explicitly holds all patterns) and
(b) the restricted master problem (integer) using **only** the columns generated by column generation to `mk.compare_variants` and compare the root dual bound, final gap, and number of nodes. We observe whether (b) reaches the same integer optimal as the compact formulation, even though it only has a fraction of the patterns generated in Section 2.

In [7]:
def build_rmp_ip():
    cols = res["columns"]
    m = Model("cutting_stock_rmp_ip")
    lam = [m.addVar(vtype="I", lb=0, name=f"lam_{p}") for p in range(len(cols))]
    for i in range(cs.N_ITEMS):
        m.addCons(quicksum(cols[p][i] * lam[p] for p in range(len(cols))) >= rhs[i],
                  name=f"demand_{i}")
    m.setObjective(quicksum(lam), "minimize")
    m.data = dict(lam=lam, cols=cols)
    return m

df = mk.compare_variants(
    {"compact(全パターン陽に列挙)": cs.build_compact,
     f"RMP-IP(列生成の{gen}列だけ)": build_rmp_ip},
    time_limit=20)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,compact(全パターン陽に列挙),23.52,24.0,0.0,5231
1,RMP-IP(列生成の13列だけ),24.00,24.0,0.0,1


In [8]:
compact_row = df.loc[df["variant"] == "compact(全パターン陽に列挙)"].iloc[0]
rmp_row = df.loc[df["variant"] == f"RMP-IP(列生成の{gen}列だけ)"].iloc[0]

labels = ["compact\n(全パターン)", f"RMP-IP\n({gen}列)"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [compact_row["root_dual"], rmp_row["root_dual"]], lambda v: f"{v:.2f}")
add_bars(2, [compact_row["final_gap"]*100, rmp_row["final_gap"]*100], lambda v: f"{v:.1f}%")
add_bars(3, [compact_row["nodes"], rmp_row["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="列生成の before / after(compact vs 生成済み列だけのRMP-IP)",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [9]:
print(f"ルート双対境界 : compact {compact_row['root_dual']:.2f}  vs  RMP-IP {rmp_row['root_dual']:.2f}")
print(f"最終gap        : compact {compact_row['final_gap']:.1%}  vs  RMP-IP {rmp_row['final_gap']:.1%}")
print(f"ノード数       : compact {int(compact_row['nodes'])}  vs  RMP-IP {int(rmp_row['nodes'])}")
print(f"使用パターン数 : compact {total_patterns}(全列挙)  vs  "
      f"RMP-IP {gen}({gen/total_patterns*100:.1f}%、pricingで生成した分だけ)")

ルート双対境界 : compact 23.52  vs  RMP-IP 24.00
最終gap        : compact 0.0%  vs  RMP-IP 0.0%
ノード数       : compact 5231  vs  RMP-IP 1
使用パターン数 : compact 131(全列挙)  vs  RMP-IP 13(9.9%、pricingで生成した分だけ)


## Summary

- **The strength of the LP bound itself is equivalent** (the GG bound and the material lower bound are almost identical). The value of column generation is not in "strengthening the LP," but in **being able to implicitly handle only the necessary columns via pricing, without enumerating an exponential number of patterns**. In this subject, it reached the best LP bound by generating only a tiny fraction of all feasible patterns (see the output at the end of Section 2).
- The restricted master problem using only generated columns (RMP-IP) can be solved with far fewer nodes than the symmetric massive compact formulation (Section 4). Reformulating in units of patterns intrinsically eliminates roll-swapping symmetry from the start.

### Why SCIP Doesn't Do It Automatically

SCIP handles the columns (variables) of a given model as they are. The reformulation to "dynamically generate patterns as columns" cannot be discovered within the scope of presolve because the model's structure (what is a variable and what is a constraint) is already fixed the moment the compact formulation is written. Column generation is a division of labor where the modeler designs the "master problem" and the "pricing problem," and this is not the kind of thing that can be automatically detected by diagnostics (although it is suggested as a recipe for `decomposable`).

### When It Doesn't Work / Cautions

- **price_and_branch has no optimality guarantee**. Do not declare it "optimal" unless you confirm `lp_lb == int_obj` (there is a counterexample in FINDINGS §4: in a small-scale test, price_and_branch=5 while the true optimal of the fully enumerated ILP=4).
- For degenerate problems (where the dual $\pi$ oscillates wildly between iterations), **Wentges stabilization** (`alpha>0`) is effective. Smoothing toward the center, such as `mk.column_generation(..., alpha=0.5)`, can reduce the number of iterations (an excessively large `alpha` can lead to non-convergence due to over-stabilization).

Related: [Playbook 6. Column Generation](../../playbook/06-column-generation.md) /
API [`mk.column_generation`/`mk.price_and_branch`](../../api/frameworks.md) /
worked example: `experiments/run_colgen.py` / `run_stabilize.py` / `run_bnp.py`